## Local Aggregation & Correlation Script

In [2]:
import os
import json
import pandas as pd
from scipy.stats import pearsonr
import re
import string
from collections import Counter
from nltk.tokenize import word_tokenize

LOCAL_DIR = os.path.expanduser('~/vlm_benchmark/outputs/LLM_as_judge_evaluation/')

# --- NEW: Safe JSONL Loader ---
def load_jsonl_safe(path):
    records = []
    with open(path, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            line = line.strip()
            if not line: continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError:
                print(f"⚠️ Warning: Skipped broken line at the end of {os.path.basename(path)}")
    return records

# --- PART 1: AGGREGATION LOGIC ---

def score_judged_file(path: str) -> dict:
    """Compute aggregate judge scores from a judged JSONL file."""
    all_records = load_jsonl_safe(path)
    records = [r for r in all_records if r.get('judge_score') is not None]

    closed = [r for r in records if r.get('is_closed', False)]
    open_  = [r for r in records if not r.get('is_closed', True)]

    def avg_score(recs):
        if not recs: return None
        return round(sum(r['judge_score'] for r in recs) / len(recs), 3)

    # Binary accuracy using judge score >= 4 as correct
    def judge_accuracy(recs):
        if not recs: return None
        return round(sum(1 for r in recs if r['judge_score'] >= 4) / len(recs) * 100, 2)

    return {
        'file':            os.path.basename(path),
        'n_judged':        len(records),
        'n_failed':        sum(1 for r in all_records if r.get('judge_score') is None),
        'avg_score_all':   avg_score(records),
        'avg_score_closed':avg_score(closed),
        'avg_score_open':  avg_score(open_),
        'judge_acc_all':   judge_accuracy(records),
        'judge_acc_closed':judge_accuracy(closed),
        'judge_acc_open':  judge_accuracy(open_),
    }

# --- PART 2: CORRELATION LOGIC ---

def tokenize_answer(text):
    text = re.sub(r'\*+', '', str(text)).lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    return word_tokenize(text)

def token_f1_score(prediction, ground_truth):
    pred_tokens = tokenize_answer(prediction)
    gt_tokens   = tokenize_answer(ground_truth)
    if not pred_tokens or not gt_tokens:
        return 0.0
    pred_set = Counter(pred_tokens)
    gt_set   = Counter(gt_tokens)
    common   = sum((pred_set & gt_set).values())
    precision = common / len(pred_tokens) if pred_tokens else 0.0
    recall    = common / len(gt_tokens) if gt_tokens else 0.0
    return (2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0)

# --- EXECUTION ---

# Find all judged files in your local directory
judged_files = sorted(
    f for f in os.listdir(LOCAL_DIR)
    if f.endswith('_judged.jsonl') and 'dry_run' not in f
)

if not judged_files:
    print(f'No judged files found in {LOCAL_DIR}. Check your path!')
else:
    print('=== 1. LLM JUDGE RESULTS (score out of 5, accuracy = score >= 4) ===\n')
    rows = []
    for fname in judged_files:
        path   = os.path.join(LOCAL_DIR, fname)
        scores = score_judged_file(path)

        # Parse model and dataset from filename 
        base    = fname.replace('_judged.jsonl', '').replace('_v2', '')
        parts   = base.split('__')
        scores['model']   = parts[0].replace('_', '/', 1)
        scores['dataset'] = parts[1] if len(parts) > 1 else 'unknown'
        rows.append(scores)

    df = pd.DataFrame(rows)
    cols = ['model', 'dataset', 'n_judged', 'n_failed',
            'avg_score_all', 'avg_score_closed', 'avg_score_open',
            'judge_acc_all', 'judge_acc_closed', 'judge_acc_open']
    
    # Sort and display
    if not df.empty:
        df = df.sort_values(['dataset', 'model']).reset_index(drop=True)
        print(df[cols].to_string(index=False))

        # Save the final CSV report locally
        csv_path = os.path.join(LOCAL_DIR, 'llm_judge_results.csv')
        df[cols].to_csv(csv_path, index=False)
        print(f'\n[Success] Report saved locally to: {csv_path}\n')
        
    print('-' * 80)
    print('\n=== 2. PEARSON CORRELATION: Token F1 vs LLM Judge Score ===\n')
    print(f'{"File":<65} {"Correlation":>12} {"p-value":>10}')
    print('-' * 80)

    for fname in judged_files:
        path    = os.path.join(LOCAL_DIR, fname)
        records = load_jsonl_safe(path)
        records = [r for r in records if r.get('judge_score') is not None]

        f1_scores    = [token_f1_score(r.get('prediction', ''), r.get('ground_truth', '')) for r in records]
        judge_scores = [r['judge_score'] for r in records]

        if len(f1_scores) > 2:
            corr, pval = pearsonr(f1_scores, judge_scores)
            print(f'{fname:<65} {corr:>12.3f} {pval:>10.4f}')

=== 1. LLM JUDGE RESULTS (score out of 5, accuracy = score >= 4) ===

                                            model    dataset  n_judged  n_failed  avg_score_all  avg_score_closed  avg_score_open  judge_acc_all  judge_acc_closed  judge_acc_open
                             google/gemma-3-4b-it      okvqa       978        22          2.809               NaN           2.809          40.90               NaN           40.90
                llava-hf/llava-v1.6-mistral-7b-hf      okvqa       982        18          3.428               NaN           3.428          57.64               NaN           57.64
                             google/gemma-3-4b-it      slake      1061         0          3.281             3.550           3.107          55.14             68.99           46.20
                            google/medgemma-4b-it      slake      1061         0          4.004             4.111           3.935          73.70             83.65           67.29
FreedomIntelligence/HuatuoGPT-Visio